In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.table("bronze_supply_chain_raw")

print(df.count())
print(len(df.columns))

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 3, Finished, Available, Finished, False)

180519
57


In [2]:
silver_df = (
    df
    .withColumn(
        "order_date",
        to_timestamp("order_date_dateorders")
    )
    .withColumn(
        "shipping_date",
        to_timestamp("shipping_date_dateorders")
    )
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 4, Finished, Available, Finished, False)

In [3]:
silver_df = (
    silver_df
    .withColumn(
        "delivery_delay_days",
        col("days_for_shipping_real")
        - col("days_for_shipment_scheduled")
    )
    .withColumn(
        "is_late_delivery",
        col("late_delivery_risk")
    )
    .withColumn(
        "is_on_time_delivery",
        when(
            lower(col("delivery_status"))
            == "shipping on time",
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_cancelled",
        when(
            lower(col("delivery_status"))
            == "shipping canceled",
            1
        ).otherwise(0)
    )
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 5, Finished, Available, Finished, False)

In [4]:
silver_df = (
    silver_df
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
    .withColumn("order_day", dayofmonth("order_date"))
    .withColumn("order_week", weekofyear("order_date"))
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 6, Finished, Available, Finished, False)

In [5]:
before_count = silver_df.count()

silver_df = silver_df.dropDuplicates()

after_count = silver_df.count()

print("Before:", before_count)
print("After :", after_count)
print("Removed:", before_count - after_count)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 7, Finished, Available, Finished, False)

Before: 180519
After : 180519
Removed: 0


In [6]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_supply_chain")
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 8, Finished, Available, Finished, False)

In [7]:
spark.table("silver_supply_chain").count()

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 9, Finished, Available, Finished, False)

180519

In [8]:
silver_base = spark.table("silver_supply_chain")

order_features = (
    silver_base
    .select(
        "order_id",
        "order_date",
        "customer_id",
        "market",
        "order_region",
        "shipping_mode",
        "category_id",
        "category_name",
        "department_id",
        "department_name",
        "sales",
        "benefit_per_order",
        "order_item_quantity",
        "order_item_discount",
        "order_item_discount_rate",
        "days_for_shipment_scheduled",
        "days_for_shipping_real",
        "delivery_delay_days",
        "late_delivery_risk",
        "is_late_delivery",
        "is_on_time_delivery",
        "is_cancelled"
    )
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 10, Finished, Available, Finished, False)

In [9]:
order_features = (
    order_features
    .withColumn(
        "profit_margin_pct",
        round(
            (col("benefit_per_order") / col("sales")) * 100,
            2
        )
    )
    .withColumn(
        "high_discount_flag",
        when(
            col("order_item_discount_rate") >= 0.20,
            1
        ).otherwise(0)
    )
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 11, Finished, Available, Finished, False)

In [10]:
(
    order_features.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_order_features")
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 12, Finished, Available, Finished, False)

In [11]:
spark.table("silver_order_features").count()

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 13, Finished, Available, Finished, False)

180519

In [12]:
access_logs = spark.table(
    "bronze_access_logs_raw"
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 14, Finished, Available, Finished, False)

In [13]:
access_logs = access_logs.dropDuplicates()

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 15, Finished, Available, Finished, False)

In [14]:
(
    access_logs.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_access_logs")
)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 16, Finished, Available, Finished, False)

In [15]:
spark.table("silver_access_logs").count()

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 17, Finished, Available, Finished, False)

466728

In [16]:
spark.sql("SHOW TABLES").show(100, False)

StatementMeta(, 4ae9b202-ef86-4fe0-a9d9-795acca1d9a0, 18, Finished, Available, Finished, False)

+-------------------------------------------------+--------------------------+-----------+
|namespace                                        |tableName                 |isTemporary|
+-------------------------------------------------+--------------------------+-----------+
|ws_dataco_supply_chain.lh_dataco_supply_chain.dbo|bronze_access_logs_raw    |false      |
|ws_dataco_supply_chain.lh_dataco_supply_chain.dbo|bronze_data_dictionary_raw|false      |
|ws_dataco_supply_chain.lh_dataco_supply_chain.dbo|bronze_supply_chain_raw   |false      |
|ws_dataco_supply_chain.lh_dataco_supply_chain.dbo|silver_access_logs        |false      |
|ws_dataco_supply_chain.lh_dataco_supply_chain.dbo|silver_order_features     |false      |
|ws_dataco_supply_chain.lh_dataco_supply_chain.dbo|silver_supply_chain       |false      |
+-------------------------------------------------+--------------------------+-----------+

